# Test pystac_client from rs-client-libraries

The pystac_client.Client class is made to read the database, not to write or update it. We have added methods for that in rs_client.stac.catalog_client. 

We are testing in this notebook that they work as intended.

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Other imports
import json
import pystac
import pystac_client
from typing import Callable

In [ ]:
# Create a test collection
col_name = "test_pystac_client"
print(f"Stage items in {col_name!r}")
# NOTE: this uses:
#   - catalog_client.remove_collection
#   - catalog_client.add_collection
#   - catalog_client.get_collection
create_test_collection(col_name)

# Stage auxip products
PRODUCT_COUNT = 3
stage_test_objects(auxip_client, PRODUCT_COUNT, col_name)

In [ ]:
def assert_nominal(product_count) -> tuple[
    pystac_client.collection_client.CollectionClient, 
    list[pystac.item.Item]
]:
    """
    Assert that all methods return the expected result.
    Returns the collection and items.
    """

    landing = catalog_client.get_landing()
    print("Landing page:")
    print(json.dumps(landing, indent=2))
    assert landing

    col_ids = [col.id for col in catalog_client.get_collections()]
    print(f"Collection IDs: {col_ids}")
    assert col_name in col_ids

    collection = catalog_client.get_collection(col_name)
    print("Collection:")
    display(collection)
    assert collection

    # Check that the raw http request also returns the collection
    http_session.get(
        f"{catalog_client.href_service}/catalog/collections/{catalog_client.owner_id}:{col_name}"
    ).raise_for_status()

    items = list(catalog_client.get_items(col_name))
    print("Items:")
    display(items)
    assert len(items) == product_count

    # Check that the raw http request also returns the same items
    response = http_session.get(
        f"{catalog_client.href_service}/catalog/collections/{catalog_client.owner_id}:{col_name}/items")
    response.raise_for_status()
    ids_from_pystac_client = [item.id for item in items]
    ids_from_raw_http = [item["id"] for item in response.json()["features"]]
    assert sorted(ids_from_pystac_client) == sorted(ids_from_raw_http)

    item = catalog_client.get_item(col_name, items[0].id)
    print("Item:")
    display(item)
    assert item

    searched = catalog_client.search(collections=[col_name])
    print("Search:")
    display(searched)
    assert len(searched) == product_count

    col_queryables = catalog_client.get_collection_queryables(col_name)
    print("Collection queryables:")
    print(json.dumps(col_queryables, indent=2))
    assert col_queryables

    queryables = catalog_client.get_queryables()
    print("Queryables:")
    print(json.dumps(queryables, indent=2))
    assert queryables

    return collection, items

def assert_missing(missing_col_name: str | None = None):
    """Assert that a collection does not exist"""

    missing_col_name = missing_col_name or col_name

    def assert_exception(func_name: str, callable: Callable):
        """Assert that a function call will raise any Exception"""
        try:
            callable()
        except Exception:
            print(f"{func_name!r} failed as excepted.")
        else:
            assert False, f"{func_name!r} was expected to fail !"

    # Assert that the pystac client methods raise exceptions
    assert_exception("get_collection", lambda: catalog_client.get_collection(missing_col_name))
    assert_exception("get_items", lambda: catalog_client.get_items(missing_col_name))
    assert_exception("search", lambda: catalog_client.search(collections=[missing_col_name]))
    assert_exception("get_collection_queryables", lambda: catalog_client.get_collection_queryables(missing_col_name))

    # Assert that the collection is missing from all the returned collections
    col_ids = [col.id for col in catalog_client.get_collections()]
    assert catalog_client.full_collection_id(None, missing_col_name, "_") not in col_ids

    # Check that the raw http request does not return the collection
    response = http_session.get(
        f"{catalog_client.href_service}/catalog/collections/{catalog_client.owner_id}:{missing_col_name}")
    assert response.status_code == 404

In [ ]:
# Check nominal case
collection, items = assert_nominal(PRODUCT_COUNT)

In [ ]:
# Check missing collection case
assert_missing("no_such_collection")

In [ ]:
# Update a collection. First modify the local instance...
collection.description = "Updated description for test_pystac_client"

# ... then push it to the server
catalog_client.update_collection(collection)

# Get it back from the server and check that we have the 
# same values than the local instance.
from_server = catalog_client.get_collection(col_name)
assert from_server.description == collection.description

# We should have the same result if we request directly the server,
# without the pystac client
raw_http_get = http_session.get(
    f"{catalog_client.href_service}/catalog/collections/{catalog_client.owner_id}:{col_name}")
assert raw_http_get.json()["description"] == collection.description

In [ ]:
# Update an item. Same as the collection...
first_item = items[0]
prop_key = "test_pystac_client_property"
prop_value = "any_value"
first_item.properties[prop_key] = prop_value

catalog_client.update_item(first_item)

from_server = catalog_client.get_item(col_name, first_item.id)
assert from_server.properties[prop_key] == prop_value

raw_http_get = http_session.get(
    f"{catalog_client.href_service}/catalog/collections/{catalog_client.owner_id}:{col_name}/items/{first_item.id}")
assert raw_http_get.json()["properties"][prop_key] == prop_value

In [ ]:
# Add an item
new_item = first_item
new_item.id = "new_item"
new_item.assets = {}
catalog_client.add_item(col_name, new_item)

# Check that the pystac client returns updated responses
assert_nominal(PRODUCT_COUNT + 1)

In [ ]:
# Remove item and check again
catalog_client.remove_item(col_name, new_item.id)
assert_nominal(PRODUCT_COUNT)

In [ ]:
# Remove the collection and check again
catalog_client.remove_collection(col_name)
assert_missing()